# Limpeza e Consolidação de Dados do SISU (2013-2023)
Este notebook realiza a leitura, padronização e filtragem dos microdados do SISU para os municípios da Baixada Fluminense.


In [ ]:
%pip install -r ../requirements.txt

In [1]:
import polars as pl
import os
import glob
import unicodedata
import pyarrow.parquet as pq

# ── Normalização Unicode: remove acentos e converte para maiúsculas ──────────
def normalize_name(s: str | None) -> str | None:
    """Remove acentos diacríticos e converte para maiúsculas.
    Garante que 'Nova Iguaçu', 'NOVA IGUAÇU' e 'NOVA IGUACU' sejam todos
    equivalentes após normalização.
    """
    if s is None:
        return None
    normalized = unicodedata.normalize('NFD', str(s))
    sem_acento = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')
    return sem_acento.upper().strip()

# ── Lista de municípios da Baixada Fluminense (sem acentos, maiúsculas) ───────
# Usada APÓS normalização — não precisamos mais de variantes com/sem acento.
BAIXADA_MUNICIPIOS_NORM = [
    'BELFORD ROXO', 'DUQUE DE CAXIAS', 'GUAPIMIRIM', 'ITAGUAI',
    'JAPERI', 'MAGE', 'MESQUITA', 'NILOPOLIS', 'NOVA IGUACU',
    'PARACAMBI', 'QUEIMADOS', 'SAO JOAO DE MERITI', 'SEROPEDICA'
]

# Mapa de normalizado → nome canônico legível (Title Case com acentos corretos)
NOME_CANONICO = {
    'BELFORD ROXO':      'Belford Roxo',
    'DUQUE DE CAXIAS':   'Duque de Caxias',
    'GUAPIMIRIM':        'Guapimirim',
    'ITAGUAI':           'Itaguaí',
    'JAPERI':            'Japeri',
    'MAGE':              'Magé',
    'MESQUITA':          'Mesquita',
    'NILOPOLIS':         'Nilópolis',
    'NOVA IGUACU':       'Nova Iguaçu',
    'PARACAMBI':         'Paracambi',
    'QUEIMADOS':         'Queimados',
    'SAO JOAO DE MERITI':'São João de Meriti',
    'SEROPEDICA':        'Seropédica',
}

# ajuste de caminho para rodar dentro da pasta scripts/
raw_sisu_path = '../raw_data/dados_sisu/'


In [2]:
# Função para unificar nomes de colunas
def map_columns_polars(df: pl.DataFrame) -> pl.DataFrame:
    # Converte todas as colunas para maiúsculo para facilitar a busca
    new_cols = {c: c.upper().strip() for c in df.columns}
    df = df.rename(new_cols)
    
    # Tratamento de variações nas colunas antigas vs novas
    rename_map = {}
    cols = df.columns
    
    if 'ANO' not in cols and 'NU_ANO' in cols: rename_map['NU_ANO'] = 'ANO'
    
    if 'MUNICIPIO_CANDIDATO' not in cols:
        if 'NO_MUNICIPIO_RESIDENCIA' in cols: rename_map['NO_MUNICIPIO_RESIDENCIA'] = 'MUNICIPIO_CANDIDATO'
        elif 'NO_MUNICIPIO_CANDIDATO' in cols: rename_map['NO_MUNICIPIO_CANDIDATO'] = 'MUNICIPIO_CANDIDATO'
        
    if 'APROVADO' not in cols and 'ST_APROVADO' in cols: rename_map['ST_APROVADO'] = 'APROVADO'
    if 'MATRICULA' not in cols and 'ST_MATRICULA' in cols: rename_map['ST_MATRICULA'] = 'MATRICULA'
    if 'NOME_IES' not in cols and 'NO_IES' in cols: rename_map['NO_IES'] = 'NOME_IES'
    if 'NOME_CURSO' not in cols and 'NO_CURSO' in cols: rename_map['NO_CURSO'] = 'NOME_CURSO'
        
    if rename_map:
        df = df.rename(rename_map)
        
    # Mantém apenas as colunas que conseguimos mapear
    keep_cols = [c for c in ['ANO', 'MUNICIPIO_CANDIDATO', 'APROVADO', 'MATRICULA', 'NOME_IES', 'NOME_CURSO'] if c in df.columns]
    df = df.select(keep_cols)
    
    # Converte tudo para snake_case
    df = df.rename({c: c.lower() for c in df.columns})
    return df


In [3]:
import re
from collections import defaultdict

# ── Detecção automática de encoding do CSV ────────────────────────────────────
def detect_encoding(filepath: str) -> str:
    """Testa UTF-8 primeiro; se falhar, usa ISO-8859-1 (Latin1).
    Resolve o problema de anos onde o MEC distribuiu CSVs em UTF-8
    (ex: 2015, 2016, 2019, 2020) em vez do histórico ISO-8859-1.
    """
    for enc in ('utf-8-sig', 'utf-8', 'iso-8859-1'):
        try:
            with open(filepath, 'r', encoding=enc, errors='strict') as f:
                f.read(8192)   # lê os primeiros 8 KB como teste
            return enc
        except (UnicodeDecodeError, LookupError):
            continue
    return 'iso-8859-1'  # fallback seguro


def detect_separator(filepath: str, encoding: str) -> str:
    """Detecta o separador do CSV (pipe ou ponto-e-vírgula)."""
    with open(filepath, 'r', encoding=encoding, errors='replace') as f:
        first_line = f.readline()
    return '|' if '|' in first_line else ';'


# Agrupa arquivos por ano
files = glob.glob(os.path.join(raw_sisu_path, '*.csv'))
files_por_ano = defaultdict(list)
for file in files:
    match = re.search(r'20\d{2}', os.path.basename(file))
    if match:
        files_por_ano[match.group()].append(file)
    else:
        print(f"Aviso: Não foi possível identificar o ano no arquivo {file}")

out_dir_microdados = '../curated/parquet/sisu/microdados_por_ano/'
out_dir_agregado   = '../curated/parquet/sisu/agregado_por_ano/'
os.makedirs(out_dir_microdados, exist_ok=True)
os.makedirs(out_dir_agregado,   exist_ok=True)

agregados_totais = []

for ano, arquivos in sorted(files_por_ano.items()):
    print(f"\nIniciando processamento do ano {ano}...")
    dfs_do_ano = []

    for file in arquivos:
        print(f"  Lendo {file}...")
        try:
            # ── CORREÇÃO 1: detectar encoding real do arquivo ─────────────────
            encoding = detect_encoding(file)
            print(f"    Encoding detectado: {encoding}")

            sep = detect_separator(file, encoding)

            df = pl.read_csv(
                file,
                separator=sep,
                encoding=encoding,
                infer_schema_length=0,
                ignore_errors=True,
            )

            # Localiza a coluna de município do candidato
            col_mun_candidato = None
            for c in df.columns:
                if c.upper().strip() in ['MUNICIPIO_CANDIDATO', 'NO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_CANDIDATO']:
                    col_mun_candidato = c
                    break

            if col_mun_candidato:
                # ── CORREÇÃO 2: normalizar via Unicode antes de filtrar ────────
                # Usa map_elements para aplicar normalize_name Python a cada célula,
                # garantindo que qualquer combinação de encoding/acentuação seja
                # corretamente comparada com BAIXADA_MUNICIPIOS_NORM.
                df = df.with_columns(
                    pl.col(col_mun_candidato)
                      .cast(pl.Utf8)
                      .map_elements(normalize_name, return_dtype=pl.Utf8)
                      .alias('__mun_norm')
                )
                df = df.filter(pl.col('__mun_norm').is_in(BAIXADA_MUNICIPIOS_NORM))

                # ── CORREÇÃO 3: padronizar nome do município para forma canônica ─
                # Substitui a coluna original pelo nome legível (com acentos corretos)
                df = df.with_columns(
                    pl.col('__mun_norm')
                      .map_elements(lambda x: NOME_CANONICO.get(x, x), return_dtype=pl.Utf8)
                      .alias(col_mun_candidato)
                )
                df = df.drop('__mun_norm')

                df = map_columns_polars(df)
                dfs_do_ano.append(df)
                print(f"    Municípios encontrados: {sorted(df['municipio_candidato'].unique().to_list())}")
            else:
                print(f"  Aviso: Coluna de município não encontrada em {file}.")

        except Exception as e:
            print(f"  Erro ao processar {file}: {e}")

    if dfs_do_ano:
        try:
            df_final_ano = pl.concat(dfs_do_ano, how='diagonal_relaxed')
        except Exception:
            df_final_ano = pl.concat(dfs_do_ano, how='diagonal')

        # Tratamento de aprovados
        if 'aprovado' in df_final_ano.columns:
            df_final_ano = df_final_ano.with_columns(
                pl.col('aprovado').cast(pl.Utf8).str.to_uppercase().str.strip_chars()
            )
            df_final_ano = df_final_ano.with_columns(
                pl.when(pl.col('aprovado').is_in(['S', 'SIM', 'TRUE', '1']))
                  .then(1).otherwise(0).alias('is_aprovado')
            )
        else:
            df_final_ano = df_final_ano.with_columns(pl.lit(0).alias('is_aprovado'))

        # Tratamento de matrículas
        if 'matricula' in df_final_ano.columns:
            df_final_ano = df_final_ano.with_columns(
                pl.col('matricula').cast(pl.Utf8).str.to_uppercase().str.strip_chars()
            )
            df_final_ano = df_final_ano.with_columns(
                pl.when(pl.col('matricula').is_in(['EFETIVADA', 'SIM', 'S', 'TRUE', '1']))
                  .then(1).otherwise(0).alias('is_matriculado')
            )
        else:
            df_final_ano = df_final_ano.with_columns(pl.lit(0).alias('is_matriculado'))

        # Garante coluna ano
        if 'ano' not in df_final_ano.columns:
            df_final_ano = df_final_ano.with_columns(pl.lit(int(ano)).alias('ano'))
        else:
            df_final_ano = df_final_ano.with_columns(pl.col('ano').cast(pl.Int32, strict=False))

        # Agrupamento
        agregado = df_final_ano.group_by(['municipio_candidato', 'ano']).agg([
            pl.len().alias('total_candidatos'),
            pl.col('is_aprovado').sum().alias('total_aprovados'),
            pl.col('is_matriculado').sum().alias('total_matriculados'),
        ])
        agregado = agregado.with_columns(
            (pl.col('total_aprovados') / pl.col('total_candidatos')).alias('taxa_aprovacao')
        )
        agregados_totais.append(agregado)

        # Salva microdado e agregado do ano
        micro_path = os.path.join(out_dir_microdados, f'sisu_microdados_{ano}.parquet')
        agg_path   = os.path.join(out_dir_agregado,   f'sisu_agregado_{ano}.parquet')
        df_final_ano.write_parquet(micro_path)
        agregado.write_parquet(agg_path)
        print(f"  Ano {ano} concluído: {df_final_ano.height} registros → {micro_path}")

# ── Consolida agregado geral ──────────────────────────────────────────────────
if agregados_totais:
    try:
        df_agregado_geral = pl.concat(agregados_totais, how='diagonal_relaxed')
    except Exception:
        df_agregado_geral = pl.concat(agregados_totais, how='diagonal')

    geral_path = '../curated/parquet/sisu/dataset_sisu_municipio_ano.parquet'
    os.makedirs(os.path.dirname(geral_path), exist_ok=True)
    df_agregado_geral.write_parquet(geral_path)
    print(f"\nTodos os anos processados! Agregado geral salvo em {geral_path}")
    print(df_agregado_geral.sort(['ano', 'municipio_candidato']))
else:
    print("Nenhum dado processado — verifique se os CSVs estão em raw_sisu_path.")



Iniciando processamento do ano 2013...
  Lendo ../raw_data/dados_sisu/sisu_2013_1_CR_com_LGPD.csv...
    Encoding detectado: utf-8-sig
    Municípios encontrados: ['Belford Roxo', 'Duque de Caxias', 'Guapimirim', 'Itaguaí', 'Japeri', 'Magé', 'Mesquita', 'Nilópolis', 'Nova Iguaçu', 'Paracambi', 'Queimados', 'Seropédica', 'São João de Meriti']
  Lendo ../raw_data/dados_sisu/sisu_2013_2_CR_com_LGPD.csv...
    Encoding detectado: utf-8-sig
    Municípios encontrados: ['Belford Roxo', 'Duque de Caxias', 'Guapimirim', 'Itaguaí', 'Japeri', 'Magé', 'Mesquita', 'Nilópolis', 'Nova Iguaçu', 'Paracambi', 'Queimados', 'Seropédica', 'São João de Meriti']
  Ano 2013 concluído: 247595 registros → ../curated/parquet/sisu/microdados_por_ano/sisu_microdados_2013.parquet

Iniciando processamento do ano 2014...
  Lendo ../raw_data/dados_sisu/relatorio_chamada_regular_SISU_2014_2.csv...
    Encoding detectado: utf-8-sig
    Municípios encontrados: ['Belford Roxo', 'Duque de Caxias', 'Guapimirim', 'Itaguaí'